# Review local SLM replies with a local decision model

A workshop team answers attendee questions using its internal planning conversation. Gemma 4 E2B drafts three short replies. Nimble reviews the drafts and chooses which can be kept and which need another local pass. Both models run through Ollama.

![Sequence diagram: Gemma drafts replies, Nimble selects actions, and Python applies them.](figures/local_review_sequence.png)

## 1. Run both models locally

Install `requirements.txt` and make `gemma4:e2b` and `nimble:9b-q4_K_M` available in Ollama. The README has the setup commands.

In [1]:
import json
import os

import pandas as pd
import requests
from dotenv import load_dotenv
from IPython.display import display

load_dotenv()
OLLAMA_URL = os.getenv("OLLAMA_URL", "http://localhost:11434").rstrip("/")
SLM_MODEL = os.getenv("OLLAMA_MODEL", "gemma4:e2b")
DECISION_MODEL = os.getenv("OLLAMA_DECISION_MODEL", "nimble:9b-q4_K_M")
SLM_OPTIONS = {"temperature": 0, "seed": 42, "num_ctx": 8192, "num_predict": 2048}

pd.set_option("display.max_colwidth", None)

## 2. Give both models the same task requirements

Three chronological messages cover registration, two different reminder audiences, and an unconfirmed start time. The team needs short replies to three attendee questions.

Each question has a requirement defined below. Initial drafting, decision review, and a focused retry all use that same requirement and the complete conversation. A reply may explain that the start time remains unconfirmed; it must not invent one.

In [3]:
conversation = [{'from': 'Alex',
  'text': 'Registration is live. The speakers have received their reminder. The workshop planning '
          'notes say 10:00.'},
 {'from': 'Sam',
  'text': 'The attendee reminder uses the same copy. It is approved and scheduled, with the '
          'speaker list still waiting for final sign-off. The calendar invitation says 10:30.'},
 {'from': 'Alex',
  'text': 'I cancelled the scheduled attendee mailing. Those people should only hear from us after '
          'the speaker list is signed off. Please confirm whether the notes or the calendar have '
          'the right start time.'}]

writing_instructions = ('Write a short attendee-facing reply to each question using the internal conversation. Follow the '
 'requirement for each question. Use only information supported by the conversation. If required '
 'information is not established, say so rather than inventing it. Return one answer per question, '
 'using its id.')

questions = [{'id': 'registration',
  'question': 'Can I register for the workshop now?',
  'requirement': 'State whether registration is currently open to the attendee.'},
 {'id': 'reminder',
  'question': 'Has my attendee reminder been sent?',
  'requirement': 'State the current attendee reminder status and any remaining prerequisite for '
                 'sending.'},
 {'id': 'start_time',
  'question': 'What time should I arrive?',
  'requirement': 'State the confirmed workshop start time if established; otherwise make the lack '
                 'of confirmation clear.'}]

source = "\n".join(message["from"] + ": " + message["text"] for message in conversation)
display(pd.DataFrame(conversation))
display(pd.DataFrame(questions))

,from,text
0,Alex,Registration is live. The speakers have received their reminder. The workshop planning notes say 10:00.
1,Sam,"The attendee reminder uses the same copy. It is approved and scheduled, with the speaker list still waiting for final sign-off. The calendar invitation says 10:30."
2,Alex,I cancelled the scheduled attendee mailing. Those people should only hear from us after the speaker list is signed off. Please confirm whether the notes or the calendar have the right start time.


,id,question,requirement
0,registration,Can I register for the workshop now?,State whether registration is currently open to the attendee.
1,reminder,Has my attendee reminder been sent?,State the current attendee reminder status and any remaining prerequisite for sending.
2,start_time,What time should I arrive?,State the confirmed workshop start time if established; otherwise make the lack of confirmation clear.


## 3. Draft all three replies in one Gemma call

`draft_replies()` sends the conversation, writing instructions, questions, and requirements to Ollama's chat endpoint. Structured output returns one `id` and `answer` per question. The schema and requested record count follow the questions passed to the function.

We will use this same function again if Nimble requests a retry, passing only the selected question.

In [4]:
def draft_replies(selected_questions):
    ids = [question["id"] for question in selected_questions]
    record_schema = {
        "type": "object",
        "properties": {
            "id": {"type": "string", "enum": ids},
            "answer": {"type": "string"},
        },
        "required": ["id", "answer"],
        "additionalProperties": False,
    }
    schema = {
        "type": "object",
        "properties": {
            "answers": {"type": "array", "items": record_schema,
                        "minItems": len(ids), "maxItems": len(ids)}
        },
        "required": ["answers"],
        "additionalProperties": False,
    }
    task = {"instructions": writing_instructions, "questions": selected_questions}
    prompt = "Perform the task below.\n\n" + json.dumps({"task": task, "conversation": conversation})
    payload = {
        "model": SLM_MODEL,
        "messages": [
            {"role": "system", "content": "Return only JSON matching the supplied schema."},
            {"role": "user", "content": prompt},
        ],
        "format": schema, "stream": False, "think": False,
        "options": SLM_OPTIONS, "keep_alive": "30m",
    }
    response = requests.post(f"{OLLAMA_URL}/api/chat", json=payload, timeout=300)
    response.raise_for_status()
    data = response.json()
    if not data.get("done") or data.get("done_reason") != "stop":
        raise RuntimeError(f"Incomplete Gemma response: {data.get('done_reason')}")
    answers = json.loads(data["message"]["content"])["answers"]
    if len(answers) != len(ids) or {answer["id"] for answer in answers} != set(ids):
        raise ValueError("Expected one answer for each requested question.")
    return answers

In [5]:
drafts = draft_replies(questions)
display(pd.DataFrame(drafts))

,id,answer
0,registration,Registration is live.
1,reminder,"The attendee reminder has been sent, but the speaker list is still waiting for final sign-off before further communication can happen."
2,start_time,There is a conflict between the planning notes (10:00) and the calendar invitation (10:30) regarding the start time.


## 4. Review the three drafts in one Nimble request

The conversation is the shared `state`. Each named question contains its original question, requirement, and Gemma's actual draft. The same three action criteria apply to every draft.

- `ACCEPT`: keep the answer.
- `REVISE_LOCALLY`: the conversation has enough information to produce a corrected answer.
- `HUMAN_REVIEW`: information needed to meet the requirement is missing.

Nimble returns a typed choice for each draft through Ollama's `/v1/systemone` endpoint.

In [6]:
action_criteria = {
    "ACCEPT": "The answer is supported by the conversation and meets the question requirement.",
    "REVISE_LOCALLY": (
        "The answer needs changing, and the conversation contains enough information to produce "
        "a corrected answer that meets the question requirement."
    ),
    "HUMAN_REVIEW": (
        "The conversation does not establish the information needed to meet the question "
        "requirement; human input is needed."
    ),
}


def review_replies(drafts):
    by_id = {draft["id"]: draft["answer"] for draft in drafts}
    review_questions = {
        question["id"]: {
            "type": "choice",
            "instructions": (
                "Evaluate the proposed attendee-facing answer against the source conversation and "
                "the requirement. Select the appropriate next action.\n\n"
                f"Question: {question['question']}\nRequirement: {question['requirement']}\n"
                f"Proposed answer: {by_id[question['id']]}"
            ),
            "criteria": action_criteria,
        }
        for question in questions
    }
    payload = {"model": DECISION_MODEL, "state": source,
               "questions": review_questions, "keep_alive": "30m"}
    response = requests.post(f"{OLLAMA_URL}/v1/systemone", json=payload, timeout=300)
    response.raise_for_status()
    answers = response.json()["answers"]
    if set(answers) != set(by_id) or any(
        answer["type"] != "choice" or answer["choice"] not in action_criteria
        for answer in answers.values()
    ):
        raise ValueError("Expected one supported choice for each draft.")
    return {record_id: answer["choice"] for record_id, answer in answers.items()}


actions = review_replies(drafts)
display(pd.DataFrame([{"id": record_id, "action": action} for record_id, action in actions.items()]))

,id,action
0,registration,ACCEPT
1,reminder,REVISE_LOCALLY
2,start_time,ACCEPT


## 5. Retry only the selected question

For `REVISE_LOCALLY`, Python calls `draft_replies([question])`. Gemma receives the same conversation, writing instructions, question, and requirement, with the work list now containing just one question. The model settings and prompt template are unchanged.

An accepted answer stays unchanged. `HUMAN_REVIEW` flags the question for a person. A focused retry is used directly; there is no repeated review loop.

In [7]:
draft_by_id = {draft["id"]: draft["answer"] for draft in drafts}
results = []
for question in questions:
    record_id = question["id"]
    action = actions[record_id]
    if action == "ACCEPT":
        final_answer = draft_by_id[record_id]
    elif action == "REVISE_LOCALLY":
        final_answer = draft_replies([question])[0]["answer"]
    else:
        final_answer = "Awaiting human input"
    results.append({"id": record_id, "question": question["question"],
                    "initial_answer": draft_by_id[record_id], "action": action,
                    "final_answer": final_answer})

display(pd.DataFrame(results)[["question", "initial_answer", "action", "final_answer"]])

,question,initial_answer,action,final_answer
0,Can I register for the workshop now?,Registration is live.,ACCEPT,Registration is live.
1,Has my attendee reminder been sent?,"The attendee reminder has been sent, but the speaker list is still waiting for final sign-off before further communication can happen.",REVISE_LOCALLY,The attendee reminder was cancelled. Those people should only hear from us after the speaker list is signed off.
2,What time should I arrive?,There is a conflict between the planning notes (10:00) and the calendar invitation (10:30) regarding the start time.,ACCEPT,There is a conflict between the planning notes (10:00) and the calendar invitation (10:30) regarding the start time.


## 6. Read the result

Gemma initially says the attendee reminder "has been sent." The conversation says the speakers received their reminder, then cancels the attendee mailing. Nimble selects `REVISE_LOCALLY` for this draft and `ACCEPT` for the other two.

The focused retry returns:

> The attendee reminder was cancelled. Those people should only hear from us after the speaker list is signed off.

The registration answer is kept. The start-time reply is also kept: its requirement allows an honest explanation that the time remains unconfirmed.

The workflow made three inference calls: one Gemma drafting call, one three-question Nimble review, and one focused Gemma retry.

## Use the pattern in another task

Replace the conversation and questions, and define each question's requirements once. The local SLM drafts the answers; the decision model selects which questions need more work. Python makes the selected follow-up calls.

- [Ollama chat API](https://docs.ollama.com/api/chat)
- [Ollama decision API](https://docs.ollama.com/api/systemone)